# Part A, final: every step keeps the best option

The same pipeline as `part_a_threshold.ipynb`, with one change: every step keeps the option with the best validation score, even when it wins by less than 0.003. Model families are selected as before (leaders within 0.01 of the best), and the ensemble is kept whenever it scores higher than every single model.

The question is whether small gains that are not significant one by one add up to a better model.

**Data.** `train` is split by product into fit (80%) and validation (20%), seed 0. Every option is fitted on fit and scored on validation. The test split is read once, in the last step. Offsets of the decision rule are tuned on out-of-fold predictions of fit, 3 folds by product.

**Metrics.** macro-F1 is the main one, MAE the second.

Finished runs are stored in `results_final.csv` and are not refitted; configs shared with `part_a_threshold.ipynb` were copied from `results_threshold.csv`. Run from this folder after `python ml/prepare_splits.py`.

In [1]:
import functools
import itertools
import json
import pathlib
import re
import time

import numpy as np
import pandas as pd
import pymorphy3
import scipy.sparse as sp
from nltk.stem import snowball
from sklearn import base
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection
from sklearn import naive_bayes
from sklearn import svm
from sklearn.feature_extraction import text as sk_text

SPLITS = pathlib.Path('../../data/splits')
RESULTS = pathlib.Path('results_final.csv')
DATASETS = ['wb', 'amazon']
SEED = 0
VAL_SIZE = 0.2
CLASSES = np.arange(1, 6)
EMOJI = re.compile('[\U0001F300-\U0001FAFF☀-➿]')
LANG = {'wb': 'russian', 'amazon': 'english'}
TIE = 0.0
LEADER_GAP = 0.01
ENSEMBLE_SIZE = 3
BIAS_GRID = np.linspace(-3, 3, 61)
WEIGHT_POWER = {'balanced': 1.0, 'sqrt': 0.5, 'none': 0.0}
OBJECTIVES = {'bias': 'macro_f1', 'bias_mae': 'mae'}
BASE_FEATURES = {'text': 'full', 'word': [1, 2], 'char': None, 'min_df': 3,
                 'max_features': 200_000, 'sublinear': True, 'binary': False}
BASELINE = {'features': BASE_FEATURES, 'model': {'name': 'logreg', 'C': 2.0},
            'imbalance': 'balanced', 'decode': 'argmax'}
COLUMNS = ['dataset', 'step', 'label', 'macro_f1', 'mae', 'accuracy',
           *[f'f1_{c}' for c in CLASSES], 'fit_sec', 'key']
CHOSEN = {}

In [2]:
class Data:
    """Train and test of one platform; train is split into fit and val."""

    def __init__(self, name):
        train, test = [pd.read_csv(SPLITS / f'{name}_{part}.csv.gz',
                                   keep_default_na=False)
                       for part in ['train', 'test']]
        splitter = model_selection.GroupShuffleSplit(
            n_splits=1, test_size=VAL_SIZE, random_state=SEED)
        fit_idx, val_idx = next(splitter.split(train, groups=train.product_id))
        # Test rows go last; only the test step reads them.
        self.df = pd.concat([train, test], ignore_index=True)
        self.y = self.df.rating.astype(int).to_numpy()
        self.groups = self.df.product_id.to_numpy()
        self.parts = {'val': (fit_idx, val_idx),
                      'test': (np.arange(len(train)),
                               np.arange(len(train), len(self.df)))}


def normalize(s):
    """Lowercases, maps emoji and '!' to tokens, collapses digits."""
    s = s.lower().replace('ё', 'е')
    s = EMOJI.sub(lambda m: f' emoji{ord(m.group()):x} ', s)
    s = re.sub(r'!+', ' excl ', s)
    return re.sub(r'\d+', '0', s)


def map_words(texts, fn):
    """Applies fn to every word, computing it once per unique word."""
    tokens = texts.str.findall(r'\w+')
    vocab = {w: fn(w) for w in set(itertools.chain.from_iterable(tokens))}
    return tokens.map(lambda ts: ' '.join(vocab[w] for w in ts))


@functools.cache
def make_text(dataset, variant):
    """Returns the review text as is, normalized, stemmed or lemmatized."""
    full = DATA[dataset].df.full_text
    if variant == 'full':
        return full
    if variant == 'clean':
        return full.map(normalize)
    if variant == 'stem':
        stem = snowball.SnowballStemmer(LANG[dataset]).stem
        return map_words(make_text(dataset, 'clean'), stem)
    morph = pymorphy3.MorphAnalyzer()
    return map_words(make_text(dataset, 'clean'),
                     lambda w: morph.parse(w)[0].normal_form)


DATA = {name: Data(name) for name in DATASETS}
pd.DataFrame({name: {part: len(idx[1]) for part, idx in d.parts.items()}
              | {'fit': len(d.parts['val'][0])}
              for name, d in DATA.items()})

,wb,amazon
val,50138,46766
test,58127,60736
fit,191457,192532


In [3]:
@functools.lru_cache(maxsize=2)
def features(dataset, spec_json, part):
    """Fits TF-IDF on the training side of a split, returns both sides."""
    spec = json.loads(spec_json)
    texts = make_text(dataset, spec['text'])
    tr, te = DATA[dataset].parts[part]
    common = dict(min_df=spec['min_df'], sublinear_tf=spec['sublinear'],
                  binary=spec['binary'], dtype=np.float32)
    vectorizers = []
    if spec['word']:
        vectorizers.append(sk_text.TfidfVectorizer(
            ngram_range=tuple(spec['word']),
            max_features=spec['max_features'], **common))
    if spec['char']:
        vectorizers.append(sk_text.TfidfVectorizer(
            analyzer='char_wb', ngram_range=tuple(spec['char']),
            max_features=300_000, **common))
    x_tr = sp.hstack([v.fit_transform(texts.iloc[tr]) for v in vectorizers])
    x_te = sp.hstack([v.transform(texts.iloc[te]) for v in vectorizers])
    return x_tr.tocsr(), x_te.tocsr()

In [4]:
class NBLogReg(base.BaseEstimator):
    """One-vs-rest logistic regressions on NB log-count-ratio features."""

    def __init__(self, C=1.0, alpha=1.0):
        self.C = C
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        """Fits one binary model per class."""
        self.classes_ = np.unique(y)
        self.ratios_, self.models_ = [], []
        for c in self.classes_:
            p = self.alpha + np.asarray(x[y == c].sum(axis=0)).ravel()
            q = self.alpha + np.asarray(x[y != c].sum(axis=0)).ravel()
            r = np.log((p / p.sum()) / (q / q.sum())).astype(np.float32)
            model = linear_model.LogisticRegression(C=self.C, max_iter=1000)
            model.fit(x.multiply(r).tocsr(), (y == c).astype(int),
                      sample_weight=sample_weight)
            self.ratios_.append(r)
            self.models_.append(model)
        return self

    def decision_function(self, x):
        """Returns per-class margins."""
        return np.column_stack([
            m.decision_function(x.multiply(r).tocsr())
            for m, r in zip(self.models_, self.ratios_)])

    def predict(self, x):
        """Returns the class with the largest margin."""
        return self.classes_[self.decision_function(x).argmax(axis=1)]


class Ensemble(base.BaseEstimator):
    """Mean of member scores scaled to unit spread; members keep weights."""

    def __init__(self, members=()):
        self.members = members

    def fit(self, x, y, sample_weight=None):
        """Fits every member with its own class-weight power."""
        self.models_, self.scales_ = [], []
        for spec in self.members:
            model = make_model({k: v for k, v in spec.items() if k != 'power'})
            model.fit(x, y, sample_weight=class_weights(y, spec['power']))
            self.models_.append(model)
            self.scales_.append(scores(model, x).std())
        return self

    def decision_function(self, x):
        """Returns the mean of scaled member scores."""
        return np.mean([scores(m, x) / s
                        for m, s in zip(self.models_, self.scales_)], axis=0)


MODELS = {
    'logreg': lambda **p: linear_model.LogisticRegression(max_iter=1000, **p),
    'linsvc': svm.LinearSVC,
    'ridge_clf': linear_model.RidgeClassifier,
    'nbsvm': NBLogReg,
    'sgd': lambda **p: linear_model.SGDClassifier(
        loss='log_loss', random_state=SEED, **p),
    'cnb': naive_bayes.ComplementNB,
    'mnb': naive_bayes.MultinomialNB,
    'ensemble': Ensemble,
}
# Models with probabilities, so the median rule applies to them.
PROBA = {'logreg', 'sgd', 'cnb', 'mnb'}


def make_model(spec):
    """Creates an unfitted model from {'name': ..., **params}."""
    params = {k: v for k, v in spec.items() if k != 'name'}
    return MODELS[spec['name']](**params)


def class_weights(y, power):
    """Returns balanced class weights raised to a power."""
    counts = pd.Series(y).value_counts()
    return ((len(y) / (len(CLASSES) * counts)).reindex(y).to_numpy())**power

In [5]:
def scores(model, x):
    """Continuous output: log-probabilities or margins."""
    if hasattr(model, 'predict_proba'):
        return np.log(np.clip(model.predict_proba(x), 1e-12, None))
    return model.decision_function(x)


def median_decode(log_proba):
    """Returns the median of the predicted distribution (MAE-optimal)."""
    cdf = np.cumsum(np.exp(log_proba), axis=1)
    return CLASSES[(cdf >= 0.5).argmax(axis=1)]


def tune_bias(s, y, objective):
    """Finds per-class score offsets maximizing macro-F1 or minimizing MAE."""
    def gain(offsets):
        pred = CLASSES[(s + offsets).argmax(1)]
        if objective == 'mae':
            return -metrics.mean_absolute_error(y, pred)
        return metrics.f1_score(y, pred, average='macro')

    bias = np.zeros(s.shape[1])
    for _, k in itertools.product(range(3), range(s.shape[1])):
        bias[k] = max(BIAS_GRID, key=lambda b: gain(
            np.r_[bias[:k], b, bias[k + 1:]]))
    return bias


def oof_scores(model, x, y, weight, groups):
    """Returns out-of-fold scores on the training side, 3 folds by product."""
    splitter = model_selection.GroupKFold(n_splits=3)
    folds = list(splitter.split(x, y, groups))
    parts = [
        scores(base.clone(model).fit(x[tr], y[tr], sample_weight=weight[tr]),
               x[te]) for tr, te in folds]
    order = np.concatenate([te for _, te in folds])
    return np.concatenate(parts)[np.argsort(order)]


def predict(cfg, x_tr, y_tr, groups, x_te):
    """Fits the model and decodes predictions with the configured rule."""
    model, decode = make_model(cfg['model']), cfg['decode']
    weight = class_weights(y_tr, WEIGHT_POWER[cfg['imbalance']])
    if decode in OBJECTIVES:
        offsets = tune_bias(oof_scores(model, x_tr, y_tr, weight, groups),
                            y_tr, OBJECTIVES[decode])
    model.fit(x_tr, y_tr, sample_weight=weight)
    if decode == 'argmax':
        return model.predict(x_te)
    s = scores(model, x_te)
    if decode == 'median':
        return median_decode(s)
    return CLASSES[(s + offsets).argmax(axis=1)]


def evaluate(y, pred):
    """Returns macro-F1, MAE, accuracy and per-class F1."""
    per_class = metrics.f1_score(
        y, pred, labels=CLASSES, average=None, zero_division=0)
    return {'macro_f1': metrics.f1_score(y, pred, average='macro'),
            'mae': metrics.mean_absolute_error(y, pred),
            'accuracy': metrics.accuracy_score(y, pred),
            **{f'f1_{c}': v for c, v in zip(CLASSES, per_class)}}

In [6]:
def key_of(cfg):
    """Returns the identity of a config: features, model, weights, rule."""
    return json.dumps(cfg, sort_keys=True)


def run(dataset, step_name, options):
    """Fits unlogged (label, config) options on validation, returns rows."""
    data = DATA[dataset]
    tr, te = data.parts['val']
    for label, cfg in options:
        results = pd.read_csv(RESULTS)
        if ((results.dataset == dataset) & (results.key == key_of(cfg))).any():
            continue
        spec = json.dumps(cfg['features'], sort_keys=True)
        x_tr, x_te = features(dataset, spec, 'val')
        start = time.time()
        pred = predict(cfg, x_tr, data.y[tr], data.groups[tr], x_te)
        row = {'dataset': dataset, 'step': step_name, 'label': label,
               **evaluate(data.y[te], pred),
               'fit_sec': round(time.time() - start, 1), 'key': key_of(cfg)}
        pd.DataFrame([row], columns=COLUMNS).to_csv(
            RESULTS, mode='a', header=False, index=False)
        print(dataset, step_name, label, round(row['macro_f1'], 4),
              round(row['mae'], 4), flush=True)
    results = pd.read_csv(RESULTS)
    rows = results[results.dataset == dataset].set_index('key')
    rows = rows.loc[[key_of(cfg) for _, cfg in options]].reset_index()
    return rows.assign(label=[label for label, _ in options])


def step(step_name, dataset, metric, options):
    """Runs options, keeps the first one within TIE of the best."""
    rows = run(dataset, step_name, options)
    sign = -1 if metric == 'mae' else 1
    score = sign * rows[metric].to_numpy()
    chosen = int(np.flatnonzero(score >= score.max() - TIE)[0])
    CHOSEN[step_name, dataset] = options[chosen][1]
    return rows.assign(step=step_name, dataset=dataset,
                       chosen=np.where(np.arange(len(rows)) == chosen, '✓', ''))


def show(tables):
    """Puts step tables of both datasets side by side, in option order."""
    rows = pd.concat(tables)
    order = rows[['step', 'label']].drop_duplicates()
    table = rows.pivot_table(index=['step', 'label'], columns='dataset',
                             values=['macro_f1', 'mae', 'chosen'],
                             aggfunc='first')
    table = table.reindex(pd.MultiIndex.from_frame(order))
    return table[['macro_f1', 'mae', 'chosen']].round(4)


def changed(cfg, **feature_changes):
    """Returns a config with some feature settings changed."""
    return {**cfg, 'features': {**cfg['features'], **feature_changes}}


def model_cfg(dataset, name, key, value, imbalance, decode):
    """Returns a config on the chosen features of a dataset."""
    return {'features': FEATURES[dataset], 'model': {'name': name, key: value},
            'imbalance': imbalance, 'decode': decode}

## Step 0. Baseline
TF-IDF of words and bigrams, logistic regression C=2 with balanced class weights, argmax.

In [7]:
# The reference point for every later step.
show([step('0_baseline', d, 'macro_f1', [('baseline', BASELINE)])
      for d in DATASETS])

macro_f1             mae         chosen   
dataset               amazon      wb  amazon      wb amazon wb
step       label                                              
0_baseline baseline   0.5224  0.4574  0.3691  0.3713      ✓  ✓

## Step 1. Features
The baseline model is kept fixed while the text representation changes. Each sub-step starts from the choice of the previous one.

### 1a. N-grams

In [8]:
show([step('1a_ngrams', d, 'macro_f1', [
    # Single words.
    ('words', changed(BASELINE, word=[1, 1])),
    # Words and bigrams.
    ('words + bigrams', BASELINE),
    # Words, bigrams and trigrams.
    ('words + bigrams + trigrams', changed(BASELINE, word=[1, 3])),
    # Character 2-5 grams inside word boundaries.
    ('char 2-5', changed(BASELINE, word=None, char=[2, 5])),
    # Words, bigrams and character 2-5 grams.
    ('words + bigrams + char 2-5', changed(BASELINE, char=[2, 5])),
]) for d in DATASETS])

macro_f1             mae         chosen  \
dataset                                amazon      wb  amazon      wb amazon   
step      label                                                                
1a_ngrams words                        0.4920  0.4322  0.4493  0.4440          
          words + bigrams              0.5224  0.4574  0.3691  0.3713          
          words + bigrams + trigrams   0.5237  0.4581  0.3627  0.3670      ✓   
          char 2-5                     0.5006  0.4431  0.4342  0.4161          
          words + bigrams + char 2-5   0.5212  0.4595  0.3709  0.3609          

                                         
dataset                              wb  
step      label                          
1a_ngrams words                          
          words + bigrams                
          words + bigrams + trigrams     
          char 2-5                       
          words + bigrams + char 2-5  ✓

Same runs as in `part_a_final.ipynb`, but the best option is kept: words + bigrams + character n-grams 2–5 on WB (0.4595, +0.0021) and words up to trigrams on Amazon (0.5237, +0.0013).

### 1b. Text preprocessing

In [9]:
def preprocessing_options(dataset):
    """Returns the text variants on top of the chosen n-grams."""
    prev = CHOSEN['1a_ngrams', dataset]
    options = [
        # Text as written.
        ('raw text', prev),
        # Lowercase, ё, emoji and '!' as tokens, digits collapsed.
        ('normalized', changed(prev, text='clean')),
        # Snowball stems of the normalized text.
        ('stems', changed(prev, text='stem')),
    ]
    # Pymorphy lemmas exist only for Russian.
    lemmas = [('lemmas', changed(prev, text='lemma'))]
    return options + lemmas * (dataset == 'wb')


show([step('1b_text', d, 'macro_f1', preprocessing_options(d))
      for d in DATASETS])

macro_f1             mae         chosen   
dataset              amazon      wb  amazon      wb amazon wb
step    label                                                
1b_text raw text     0.5237  0.4595  0.3627  0.3609         ✓
        normalized   0.5253  0.4593  0.3612  0.3604      ✓   
        stems        0.5210  0.4584  0.3647  0.3671          
        lemmas          NaN  0.4573     NaN  0.3673    NaN

WB keeps the raw text: normalization (0.4593), stems and lemmas are slightly below it on top of character n-grams. Amazon takes normalized text, 0.5253 (+0.0016).

### 1c. TF-IDF scheme

In [10]:
def tfidf_options(dataset):
    """Returns term-frequency and vocabulary variants."""
    prev = CHOSEN['1b_text', dataset]
    return [
        # Sublinear tf, 200k features, min_df=3.
        ('sublinear tf, 200k', prev),
        # Raw term counts.
        ('raw tf', changed(prev, sublinear=False)),
        # Presence only.
        ('binary tf', changed(prev, binary=True)),
        # Vocabulary cut to 50k.
        ('50k features', changed(prev, max_features=50_000)),
        # Full vocabulary with min_df=2.
        ('full vocabulary', changed(prev, max_features=None, min_df=2)),
    ]


show([step('1c_tfidf', d, 'macro_f1', tfidf_options(d)) for d in DATASETS])

macro_f1             mae         chosen   
dataset                       amazon      wb  amazon      wb amazon wb
step     label                                                        
1c_tfidf sublinear tf, 200k   0.5253  0.4595  0.3612  0.3609          
         raw tf               0.5237  0.4588  0.3633  0.3623          
         binary tf            0.5224  0.4591  0.3650  0.3598          
         50k features         0.5121  0.4542  0.3900  0.3699          
         full vocabulary      0.5345  0.4604  0.3472  0.3568      ✓  ✓

In [11]:
# Features used by every later step.
FEATURES = {d: CHOSEN['1c_tfidf', d]['features'] for d in DATASETS}
pd.DataFrame(FEATURES)

,wb,amazon
text,full,clean
word,"[1, 2]","[1, 3]"
char,"[2, 5]",None
min_df,2,2
max_features,None,None
sublinear,True,True
binary,False,False


Both datasets take the full vocabulary with min_df=2: +0.0009 on WB and +0.0092 on Amazon, where trigrams make the vocabulary much larger than 200k. Raw or binary term frequency and a 50k vocabulary are lower.

## Step 2. Model families
Every family gets its own regularization grid, from the strongest regularization to the weakest, with balanced class weights and argmax as in the baseline. Families within 0.01 of the best one become leaders: later steps can still change their order.

In [12]:
# Family: (parameter, grid from the strongest regularization).
MODEL_GRID = {
    # Logistic regression.
    'logreg': ('C', [0.25, 0.5, 1.0, 2.0, 4.0]),
    # Linear SVM.
    'linsvc': ('C', [0.01, 0.03, 0.1, 0.3]),
    # Ridge classifier.
    'ridge_clf': ('alpha', [30.0, 10.0, 3.0, 1.0]),
    # Logistic regression on naive Bayes ratios (NB-SVM).
    'nbsvm': ('C', [0.3, 1.0, 3.0]),
    # SGD with log loss.
    'sgd': ('alpha', [3e-5, 1e-5, 3e-6]),
    # Complement naive Bayes.
    'cnb': ('alpha', [1.0, 0.3, 0.1]),
    # Multinomial naive Bayes.
    'mnb': ('alpha', [0.3, 0.1, 0.03]),
}

family_tables = [
    step(f'2_{name}', d, 'macro_f1',
         [(f'{key}={v}', model_cfg(d, name, key, v, 'balanced', 'argmax'))
          for v in grid])
    for d, (name, (key, grid)) in itertools.product(
        DATASETS, MODEL_GRID.items())]
show(family_tables)

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
2_logreg    C=0.25        0.5270  0.4651  0.3952  0.3781         ✓
            C=0.5         0.5327  0.4645  0.3766  0.3698          
            C=1.0         0.5337  0.4629  0.3608  0.3624          
            C=2.0         0.5345  0.4604  0.3472  0.3568      ✓   
            C=4.0         0.5286  0.4568  0.3408  0.3535          
2_linsvc    C=0.01        0.5232  0.4672  0.3922  0.3707          
            C=0.03        0.5298  0.4674  0.3685  0.3609         ✓
            C=0.1         0.5337  0.4613  0.3489  0.3594      ✓   
            C=0.3         0.5261  0.4549  0.3433  0.3592          
2_ridge_clf alpha=30.0    0.5267  0.4655  0.3786  0.3639         ✓
            alpha=10.0    0.5319  0.4629  0.3574  0.3612      ✓   
            alpha=3.0     0.5288  0.4558  0.3446  0.3647          
            alpha=1.0     0.5150  0.4457  0.3464  0.3754          
2_nbsvm     C=0.3         0.5142  0.4727  0.4202  0.3528         ✓
            C=1.0         0.5177  0.4677  0.3940  0.3447          
            C=3.0         0.5195  0.4617  0.3819  0.3430      ✓   
2_sgd       alpha=3e-05   0.5215  0.4665  0.4046  0.3786         ✓
            alpha=1e-05   0.5319  0.4659  0.3767  0.3681          
            alpha=3e-06   0.5349  0.4592  0.3455  0.3624      ✓   
2_cnb       alpha=1.0     0.5096  0.4492  0.4092  0.3985      ✓  ✓
            alpha=0.3     0.5010  0.4439  0.4059  0.3967          
            alpha=0.1     0.4886  0.4395  0.4110  0.3970          
2_mnb       alpha=0.3     0.5092  0.4445  0.4274  0.4424          
            alpha=0.1     0.5099  0.4469  0.3920  0.4252      ✓   
            alpha=0.03    0.5036  0.4479  0.3713  0.4051         ✓

In [13]:
# Best setting of every family and the leaders.
families = pd.concat(family_tables)
families = families[families.chosen == '✓'].assign(
    family=lambda f: f.step.str.removeprefix('2_'))
families['gap'] = (families.macro_f1
                   - families.groupby('dataset').macro_f1.transform('max'))
families['leader'] = families.gap > -LEADER_GAP
LEADERS = {d: families[(families.dataset == d) & families.leader]
           .sort_values('macro_f1', ascending=False).family.tolist()
           for d in DATASETS}
families.round(4).pivot(index='family', columns='dataset',
                        values=['label', 'macro_f1', 'gap', 'leader'])

label              macro_f1             gap         leader  \
dataset         amazon           wb   amazon      wb  amazon      wb amazon   
family                                                                        
cnb          alpha=1.0    alpha=1.0   0.5096  0.4492 -0.0252 -0.0236  False   
linsvc           C=0.1       C=0.03   0.5337  0.4674 -0.0011 -0.0053   True   
logreg           C=2.0       C=0.25   0.5345  0.4651 -0.0004 -0.0076   True   
mnb          alpha=0.1   alpha=0.03   0.5099  0.4479  -0.025 -0.0249  False   
nbsvm            C=3.0        C=0.3   0.5195  0.4727 -0.0153     0.0  False   
ridge_clf   alpha=10.0   alpha=30.0   0.5319  0.4655  -0.003 -0.0072   True   
sgd        alpha=3e-06  alpha=3e-05   0.5349  0.4665     0.0 -0.0062   True   

                  
dataset       wb  
family            
cnb        False  
linsvc      True  
logreg      True  
mnb        False  
nbsvm       True  
ridge_clf   True  
sgd         True

NB-SVM is the best family on WB (0.4727), the other four linear families are within 0.008. On Amazon NB-SVM falls 0.015 behind and is dropped; the leaders are SGD, logistic regression, the linear SVM and ridge. Naive Bayes is dropped on both.

## Steps 3–4. Tuning the leaders for macro-F1
Step 3 picks class weights and the decision rule together, at the parameter from step 2, because they interact: without weights argmax almost never predicts 2★, while offsets fix that on their own. Options go from the step 2 setting (balanced, argmax) to offsets without weights. Step 4 then reruns the regularization grid at the chosen pair.

In [14]:
# Weights and rule pairs, from the step 2 setting.
F1_PAIRS = [(w, r) for r in ['argmax', 'bias'] for w in WEIGHT_POWER]

show([step(f'3_{name}', d, 'macro_f1',
           [(f'{w}, {r}', {**CHOSEN[f'2_{name}', d], 'imbalance': w,
                           'decode': r}) for w, r in F1_PAIRS])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                        amazon      wb  amazon      wb amazon wb
step        label                                                      
3_nbsvm     balanced, argmax      NaN  0.4727     NaN  0.3528    NaN   
            sqrt, argmax          NaN  0.4664     NaN  0.2989    NaN   
            none, argmax          NaN  0.4034     NaN  0.3344    NaN   
            balanced, bias        NaN  0.4816     NaN  0.3068    NaN   
            sqrt, bias            NaN  0.4805     NaN  0.3028    NaN   
            none, bias            NaN  0.4833     NaN  0.3060    NaN  ✓
3_linsvc    balanced, argmax   0.5337  0.4674  0.3489  0.3609          
            sqrt, argmax       0.5191  0.4573  0.3111  0.3112          
            none, argmax       0.4538  0.3911  0.3455  0.3467          
            balanced, bias     0.5399  0.4738  0.3311  0.3305          
            sqrt, bias         0.5417  0.4810  0.3144  0.3140      ✓  ✓
            none, bias         0.5408  0.4766  0.3101  0.3169          
3_sgd       balanced, argmax   0.5349  0.4665  0.3455  0.3786          
            sqrt, argmax       0.5291  0.4580  0.3064  0.3162          
            none, argmax       0.4851  0.3892  0.3275  0.3577          
            balanced, bias     0.5401  0.4789  0.3224  0.3235          
            sqrt, bias         0.5435  0.4795  0.3156  0.3243      ✓  ✓
            none, bias         0.5429  0.4791  0.3145  0.3172          
3_ridge_clf balanced, argmax   0.5319  0.4655  0.3574  0.3639          
            sqrt, argmax       0.5082  0.4444  0.3215  0.3197          
            none, argmax       0.4197  0.3630  0.3878  0.3836          
            balanced, bias     0.5386  0.4729  0.3351  0.3299          
            sqrt, bias         0.5410  0.4761  0.3211  0.3144      ✓  ✓
            none, bias         0.5335  0.4714  0.3299  0.3202          
3_logreg    balanced, argmax   0.5345  0.4651  0.3472  0.3781          
            sqrt, argmax       0.5331  0.4710  0.3083  0.3088          
            none, argmax       0.5047  0.4198  0.3141  0.3271          
            balanced, bias     0.5385  0.4774  0.3271  0.3229          
            sqrt, bias         0.5395  0.4812  0.3238  0.3116          
            none, bias         0.5444  0.4850  0.3129  0.3096      ✓  ✓

In [15]:
# Regularization at the chosen weights and rule.
show([step(f'4_{name}', d, 'macro_f1',
           [(f'{MODEL_GRID[name][0]}={v}',
             {**CHOSEN[f'3_{name}', d],
              'model': {'name': name, MODEL_GRID[name][0]: v}})
            for v in MODEL_GRID[name][1]])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
4_nbsvm     C=0.3            NaN  0.4833     NaN  0.3060    NaN   
            C=1.0            NaN  0.4845     NaN  0.3010    NaN  ✓
            C=3.0            NaN  0.4721     NaN  0.3019    NaN   
4_linsvc    C=0.01        0.5238  0.4782  0.3515  0.3307          
            C=0.03        0.5390  0.4810  0.3286  0.3140         ✓
            C=0.1         0.5417  0.4758  0.3144  0.3049      ✓   
            C=0.3         0.5349  0.4649  0.3217  0.3261          
4_sgd       alpha=3e-05   0.5323  0.4795  0.3476  0.3243         ✓
            alpha=1e-05   0.5424  0.4786  0.3262  0.3081          
            alpha=3e-06   0.5435  0.4716  0.3156  0.3117      ✓   
4_ridge_clf alpha=30.0    0.5291  0.4761  0.3398  0.3144         ✓
            alpha=10.0    0.5410  0.4751  0.3211  0.3171      ✓   
            alpha=3.0     0.5365  0.4700  0.3200  0.3205          
            alpha=1.0     0.5231  0.4557  0.3304  0.3402          
4_logreg    C=0.25        0.5390  0.4850  0.3374  0.3096          
            C=0.5         0.5413  0.4852  0.3318  0.3091         ✓
            C=1.0         0.5455  0.4843  0.3200  0.3028      ✓   
            C=2.0         0.5444  0.4762  0.3129  0.3109          
            C=4.0         0.5386  0.4674  0.3189  0.3194

Offsets win for every leader. Logistic regression without weights becomes the best model for macro-F1: 0.4852 on WB (C=0.5) and 0.5455 on Amazon (C=1). NB-SVM on WB is 0.0007 behind.

## Steps 5–6. Tuning the leaders for MAE
The same two steps with MAE as the target. Rules: argmax, the median of the predicted distribution (only for models with probabilities) and offsets tuned to minimize MAE.

In [16]:
def mae_pairs(name):
    """Returns weights and MAE rule pairs, simplest rule first."""
    rules = ['argmax'] + ['median'] * (name in PROBA) + ['bias_mae']
    return [(w, r) for r in rules for w in WEIGHT_POWER]


show([step(f'5_{name}', d, 'mae',
           [(f'{w}, {r}', {**CHOSEN[f'2_{name}', d], 'imbalance': w,
                           'decode': r}) for w, r in mae_pairs(name)])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                          amazon      wb  amazon      wb amazon wb
step        label                                                        
5_nbsvm     balanced, argmax        NaN  0.4727     NaN  0.3528    NaN   
            sqrt, argmax            NaN  0.4664     NaN  0.2989    NaN   
            none, argmax            NaN  0.4034     NaN  0.3344    NaN   
            balanced, bias_mae      NaN  0.4566     NaN  0.2988    NaN   
            sqrt, bias_mae          NaN  0.4604     NaN  0.2901    NaN   
            none, bias_mae          NaN  0.4572     NaN  0.2892    NaN  ✓
5_linsvc    balanced, argmax     0.5337  0.4674  0.3489  0.3609          
            sqrt, argmax         0.5191  0.4573  0.3111  0.3112          
            none, argmax         0.4538  0.3911  0.3455  0.3467          
            balanced, bias_mae   0.5383  0.4688  0.3196  0.3172          
            sqrt, bias_mae       0.5360  0.4726  0.3066  0.2987          
            none, bias_mae       0.5316  0.4595  0.3052  0.2941      ✓  ✓
5_sgd       balanced, argmax     0.5349  0.4665  0.3455  0.3786          
            sqrt, argmax         0.5291  0.4580  0.3064  0.3162          
            none, argmax         0.4851  0.3892  0.3275  0.3577          
            balanced, median     0.5284  0.4190  0.3740  0.4518          
            sqrt, median         0.5480  0.4612  0.3037  0.3138          
            none, median         0.5323  0.4285  0.2952  0.3165      ✓   
            balanced, bias_mae   0.5312  0.4759  0.3126  0.3121          
            sqrt, bias_mae       0.5293  0.4597  0.3065  0.3084         ✓
            none, bias_mae       0.5281  0.4636  0.3053  0.3087          
5_ridge_clf balanced, argmax     0.5319  0.4655  0.3574  0.3639          
            sqrt, argmax         0.5082  0.4444  0.3215  0.3197          
            none, argmax         0.4197  0.3630  0.3878  0.3836          
            balanced, bias_mae   0.5346  0.4636  0.3231  0.3187          
            sqrt, bias_mae       0.5380  0.4738  0.3146  0.3055          
            none, bias_mae       0.5205  0.4502  0.3104  0.3046      ✓  ✓
5_logreg    balanced, argmax     0.5345  0.4651  0.3472  0.3781          
            sqrt, argmax         0.5331  0.4710  0.3083  0.3088          
            none, argmax         0.5047  0.4198  0.3141  0.3271          
            balanced, median     0.5304  0.4394  0.3658  0.4228          
            sqrt, median         0.5471  0.4760  0.3043  0.3079          
            none, median         0.5357  0.4626  0.2894  0.2919      ✓  ✓
            balanced, bias_mae   0.5308  0.4745  0.3145  0.3081          
            sqrt, bias_mae       0.5372  0.4732  0.3060  0.3011          
            none, bias_mae       0.5387  0.4704  0.3020  0.2965

In [17]:
# Regularization at the chosen weights and MAE rule.
show([step(f'6_{name}', d, 'mae',
           [(f'{MODEL_GRID[name][0]}={v}',
             {**CHOSEN[f'5_{name}', d],
              'model': {'name': name, MODEL_GRID[name][0]: v}})
            for v in MODEL_GRID[name][1]])
      for d in DATASETS for name in LEADERS[d]])

macro_f1             mae         chosen   
dataset                   amazon      wb  amazon      wb amazon wb
step        label                                                 
6_nbsvm     C=0.3            NaN  0.4572     NaN  0.2892    NaN   
            C=1.0            NaN  0.4562     NaN  0.2892    NaN  ✓
            C=3.0            NaN  0.4543     NaN  0.2948    NaN   
6_linsvc    C=0.01        0.5020  0.4458  0.3557  0.3071          
            C=0.03        0.5140  0.4595  0.3177  0.2941          
            C=0.1         0.5316  0.4685  0.3052  0.2924         ✓
            C=0.3         0.5235  0.4600  0.3024  0.2992      ✓   
6_sgd       alpha=3e-05   0.4552  0.4597  0.3841  0.3084          
            alpha=1e-05   0.5042  0.4719  0.3246  0.3005         ✓
            alpha=3e-06   0.5323  0.4592  0.2952  0.3019      ✓   
6_ridge_clf alpha=30.0    0.5080  0.4502  0.3349  0.3046          
            alpha=10.0    0.5205  0.4595  0.3104  0.2968         ✓
            alpha=3.0     0.5186  0.4491  0.3055  0.3021      ✓   
            alpha=1.0     0.5163  0.4437  0.3116  0.3156          
6_logreg    C=0.25        0.4952  0.4626  0.3356  0.2919          
            C=0.5         0.5165  0.4723  0.3126  0.2822          
            C=1.0         0.5317  0.4762  0.2955  0.2805         ✓
            C=2.0         0.5357  0.4795  0.2894  0.2808      ✓   
            C=4.0         0.5375  0.4736  0.2903  0.2887

For MAE logistic regression without weights and with the median wins on both datasets: 0.2805 on WB (C=1) and 0.2894 on Amazon (C=2). Unlike offsets tuned for MAE, the median keeps 2★ predictions.

## Step 7. Ensemble
The best tuned leaders of a metric (at most three) are averaged. Each member keeps its own class weights, its scores are scaled to unit spread, and offsets for the target metric are tuned on the out-of-fold average.

In [18]:
def tuned(dataset, prefix):
    """Returns (family, config) of every tuned leader."""
    return [(name, CHOSEN[f'{prefix}_{name}', dataset])
            for name in LEADERS[dataset]]


def ensemble_cfg(dataset, prefix, metric, decode):
    """Returns an ensemble of the best tuned leaders for a metric."""
    options = tuned(dataset, prefix)
    rows = run(dataset, f'{prefix}_lookup', options)
    order = rows[metric].to_numpy().argsort()
    order = order if metric == 'mae' else order[::-1]
    members = [{**options[i][1]['model'],
                'power': WEIGHT_POWER[options[i][1]['imbalance']]}
               for i in order[:ENSEMBLE_SIZE]]
    return {'features': FEATURES[dataset],
            'model': {'name': 'ensemble', 'members': members},
            'imbalance': 'none', 'decode': decode}


show([step(f'7_ensemble_{metric}', d, metric,
           [(' + '.join(m['name'] for m in cfg['model']['members']), cfg)])
      for d in DATASETS
      for metric, prefix, decode in [('macro_f1', '4', 'bias'),
                                     ('mae', '6', 'bias_mae')]
      for cfg in [ensemble_cfg(d, prefix, metric, decode)]])

macro_f1             mae          \
dataset                                       amazon      wb  amazon      wb   
step                label                                                      
7_ensemble_macro_f1 logreg + nbsvm + linsvc      NaN  0.4826     NaN  0.3080   
7_ensemble_mae      logreg + nbsvm + linsvc      NaN  0.4720     NaN  0.2873   
7_ensemble_macro_f1 logreg + sgd + linsvc     0.5462     NaN  0.3186     NaN   
7_ensemble_mae      logreg + sgd + linsvc     0.5411     NaN  0.3002     NaN   

                                            chosen       
dataset                                     amazon   wb  
step                label                                
7_ensemble_macro_f1 logreg + nbsvm + linsvc    NaN    ✓  
7_ensemble_mae      logreg + nbsvm + linsvc    NaN    ✓  
7_ensemble_macro_f1 logreg + sgd + linsvc        ✓  NaN  
7_ensemble_mae      logreg + sgd + linsvc        ✓  NaN

The macro-F1 ensemble is better than its best member only on Amazon, 0.5462 against 0.5455. On WB it is below logistic regression (0.4826), and both MAE ensembles are worse than logistic regression with the median.

## Step 8. Final choice
For each metric the tuned leaders are listed in a fixed order of complexity, then the ensemble. Logistic regression is the standard model with calibrated probabilities. Ridge and the linear SVM have no probabilities. SGD is a stochastic approximation. NB-SVM fits one model per class on reweighted features. The same rule keeps the first option within 0.003 of the best. Measured fit times are not used: they include pauses of the machine.

In [19]:
# Families from the simplest to the most complex.
COMPLEXITY = ['cnb', 'mnb', 'logreg', 'ridge_clf', 'linsvc', 'sgd', 'nbsvm']


def final_options(dataset, prefix, metric):
    """Returns tuned leaders by complexity, then the ensemble."""
    options = sorted(tuned(dataset, prefix),
                     key=lambda option: COMPLEXITY.index(option[0]))
    return options + [('ensemble', CHOSEN[f'7_ensemble_{metric}', dataset])]


show([step(f'8_final_{metric}', d, metric, final_options(d, prefix, metric))
      for d in DATASETS
      for metric, prefix in [('macro_f1', '4'), ('mae', '6')]])

macro_f1             mae         chosen   
dataset                      amazon      wb  amazon      wb amazon wb
step             label                                               
8_final_macro_f1 logreg      0.5455  0.4852  0.3200  0.3091         ✓
                 ridge_clf   0.5410  0.4761  0.3211  0.3144          
                 linsvc      0.5417  0.4810  0.3144  0.3140          
                 sgd         0.5435  0.4795  0.3156  0.3243          
                 nbsvm          NaN  0.4845     NaN  0.3010    NaN   
                 ensemble    0.5462  0.4826  0.3186  0.3080      ✓   
8_final_mae      logreg      0.5357  0.4762  0.2894  0.2805      ✓  ✓
                 ridge_clf   0.5186  0.4595  0.3055  0.2968          
                 linsvc      0.5235  0.4685  0.3024  0.2924          
                 sgd         0.5323  0.4719  0.2952  0.3005          
                 nbsvm          NaN  0.4562     NaN  0.2892    NaN   
                 ensemble    0.5411  0.4720  0.3002  0.2873

In [20]:
# Chosen configurations.
pd.DataFrame({(d, metric): {
    'model': CHOSEN[f'8_final_{metric}', d]['model'],
    'weights': CHOSEN[f'8_final_{metric}', d]['imbalance'],
    'rule': CHOSEN[f'8_final_{metric}', d]['decode'],
    'features': CHOSEN[f'8_final_{metric}', d]['features']}
    for d in DATASETS for metric in ['macro_f1', 'mae']})

wb  \
                                                   macro_f1   
model                          {'name': 'logreg', 'C': 0.5}   
weights                                                none   
rule                                                   bias   
features  {'text': 'full', 'word': [1, 2], 'char': [2, 5...   

                                                             \
                                                        mae   
model                          {'name': 'logreg', 'C': 1.0}   
weights                                                none   
rule                                                 median   
features  {'text': 'full', 'word': [1, 2], 'char': [2, 5...   

                                                     amazon  \
                                                   macro_f1   
model     {'name': 'ensemble', 'members': [{'name': 'log...   
weights                                                none   
rule                                                   bias   
features  {'text': 'clean', 'word': [1, 3], 'char': None...   

                                                             
                                                        mae  
model                          {'name': 'logreg', 'C': 2.0}  
weights                                                none  
rule                                                 median  
features  {'text': 'clean', 'word': [1, 3], 'char': None...

Final configurations:
- WB, macro-F1: logistic regression C=0.5, no weights, offsets; words, bigrams and character n-grams, full vocabulary.
- WB, MAE: logistic regression C=1, no weights, median; same features.
- Amazon, macro-F1: ensemble of logistic regression, SGD and the linear SVM, offsets; normalized text, words up to trigrams, full vocabulary.
- Amazon, MAE: logistic regression C=2, no weights, median; same features.

On validation this beats the threshold rule of `part_a_threshold.ipynb` in every cell: macro-F1 +0.0064 / +0.0053, MAE −0.0151 / −0.0019.

## Step 9. Test
The chosen models are fitted on the whole train split and scored once on test. The baseline is shown for reference and was not used to choose anything.

In [21]:
def test_scores(dataset, cfg):
    """Fits on the whole train split, returns metrics on the test split."""
    data = DATA[dataset]
    tr, te = data.parts['test']
    spec = json.dumps(cfg['features'], sort_keys=True)
    x_tr, x_te = features(dataset, spec, 'test')
    pred = predict(cfg, x_tr, data.y[tr], data.groups[tr], x_te)
    return evaluate(data.y[te], pred)


pd.DataFrame({(d, label): test_scores(d, cfg)
              for d in DATASETS
              for label, cfg in [
                  ('baseline', BASELINE),
                  ('final for macro-F1', CHOSEN['8_final_macro_f1', d]),
                  ('final for MAE', CHOSEN['8_final_mae', d])]}).T.round(3)

macro_f1    mae  accuracy   f1_1   f1_2   f1_3  \
wb     baseline               0.457  0.368     0.764  0.549  0.164  0.290   
       final for macro-F1     0.486  0.296     0.800  0.584  0.187  0.320   
       final for MAE          0.482  0.274     0.815  0.565  0.191  0.339   
amazon baseline               0.520  0.382     0.711  0.708  0.286  0.357   
       final for macro-F1     0.548  0.333     0.751  0.716  0.335  0.385   
       final for MAE          0.536  0.306     0.767  0.720  0.299  0.381   

                            f1_4   f1_5  
wb     baseline            0.362  0.918  
       final for macro-F1  0.399  0.939  
       final for MAE       0.369  0.945  
amazon baseline            0.376  0.873  
       final for macro-F1  0.401  0.904  
       final for MAE       0.364  0.917

Test agrees with validation: macro-F1 0.486 / 0.548 against 0.485 / 0.546, MAE 0.274 / 0.306 against 0.281 / 0.289.

Compared with `part_a_threshold.ipynb` (threshold 0.003) the greedy models are better on test in every cell: macro-F1 0.486 / 0.548 against 0.482 / 0.540, MAE 0.274 / 0.306 against 0.292 / 0.309. Small gains that are not significant one by one add up, mostly through richer features: character n-grams on WB, trigrams with the full vocabulary on Amazon. The WB MAE model with the median also keeps 2★ (F1 0.191 against 0.048). The price is a slower fit: character n-grams make WB about 5 times slower.